# Crawler de Dados - Chaves na Mão

Os anúncios usados na aula foram coletados da API pública de listagem do
[Chaves na Mão](https://www.chavesnamao.com.br). Este notebook percorre esse processo passo a passo.

O código vive em `simple_crawler.py`, e é ele que este notebook importa — assim existe uma única versão
da lógica, e o que você executa aqui é exatamente o que gerou os arquivos em `dados/anuncios`.

In [ ]:
%pip install --quiet requests pydantic

In [ ]:
from simple_crawler import ChavesNaMaoCrawler

crawler = ChavesNaMaoCrawler()

## 1. Uma página da API

O site expõe a listagem em JSON. A URL combina dois níveis de navegação — o tipo de negócio (`level1`)
e a localidade (`level2`) — mais o número da página.

In [ ]:
level1 = "imoveis-a-venda"
level2 = "go-goiania"

url = f"{crawler.base_url}{crawler.base_path}?level1={level1}&level2={level2}&pg=1"
url

In [ ]:
payload = crawler.coletar_pagina(url)
list(payload.keys())

## 2. A estrutura da resposta

A resposta tem duas partes: `items`, com os anúncios da página, e `metadata`, com as informações de
paginação. É o `metadata` que diz quantas páginas existem e qual é a próxima URL.

In [ ]:
print("anúncios nesta página:", len(payload["items"]))
print("total de páginas:", payload["metadata"]["totalPages"])
print("próxima página:", payload["metadata"]["links"]["nextApiParams"])

## 3. Do JSON para o modelo

O JSON da API é aninhado e irregular: preço, área e contagem de quartos aparecem em formatos diferentes
conforme o anúncio. O `extrair_dados` achata isso em um modelo `Anuncio` do Pydantic, com tipos fixos —
e é esse formato achatado que a tabela do BigQuery espera na aula de Pub/Sub.

Nem todo item vira anúncio: os que não têm `id` são descartados.

In [ ]:
anuncios = crawler.extrair_dados(payload)
print(f"{len(anuncios)} anúncios extraídos de {len(payload['items'])} itens")

print(anuncios[0].model_dump_json(indent=2))

In [ ]:
# O modelo também tem comportamento, não só dados
anuncios[0].endereco()

## 4. Paginação

O `coletar_paginas` repete o processo acima seguindo a próxima URL de cada resposta, com uma pausa entre
as requisições. A API não deixa passar da página 100, e o método já trata esse limite.

O `ultima` abaixo está fixo em 2 de propósito: sem ele, a coleta iria até o fim das mais de mil páginas.

In [ ]:
anuncios = crawler.coletar_paginas(level1, level2, primeira=1, ultima=2)
len(anuncios)

## 5. Como os arquivos da aula foram gerados

O arquivo `dados/chavesnamao_level2.txt` tem a lista de estados e cidades usada na coleta. A função
`realizar_coleta_varias_paginas()`, no fim do `simple_crawler.py`, percorre essa lista e grava um
`.jsonl` por localidade em `dados/anuncios`, pulando os que já existem.

Foi assim que os 162 arquivos distribuídos em `dados/anuncios.zip` foram produzidos, em setembro de 2025.
Para refazer a coleta, rode o script direto pelo terminal, a partir da raiz do repositório:

```bash
python simple_crawler.py
```